## **TMDB Movie Data Analysis using Spark and APIs**


In [9]:
import os
import time
import json
import sys
import requests
from dotenv import load_dotenv
from pyspark.sql import SparkSession, functions as F
import matplotlib.pyplot as plt
from pathlib import Path


### Functions from scripts

In [8]:
sys.path.append(os.path.abspath("..")) 
from src.tmdb_client import fetch_movie_with_credits

Let us start the Spark session.

In [3]:
spark = SparkSession.builder.appName("tmdb_lab").getOrCreate()
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/30 10:28:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### 1. **Fetching the movie data from the API**

In this project, we will follow the medallion workflow to ingest, transform and analyze the movie data for the wanted ids.

In [11]:
load_dotenv()

TMDB_API_KEY = os.getenv("TMDB_API_KEY")

print(f"The TMDB API Key is fetched")

The TMDB API Key is fetched


In [4]:
# TMDB base URLs and Endpoints
BASE_URL = "https://api.themoviedb.org/3"
MOVIE_ENDPOINT = f"{BASE_URL}/movie"
CREDITS_ENDPOINT = f"{BASE_URL}/movie"

# Movie IDs to fetch
movie_ids = [
    0, 299534, 19995, 140607, 299536, 597, 135397, 420818,
    24428, 168259, 99861, 284054, 12445, 181808, 330457,
    351286, 109445, 321612, 260513
]

len(movie_ids), movie_ids[:5]


(19, [0, 299534, 19995, 140607, 299536])

#### Bronze Layer:

Let us ingest our raw data first

In [15]:
PROJECT_ROOT = Path(os.getcwd()).parent
out_dir = PROJECT_ROOT / "data" / "bronze" / "tmdb_raw_json"
out_dir.mkdir(parents=True, exist_ok=True)

movie_ids = [
    0, 299534, 19995, 140607, 299536, 597, 135397, 420818,
    24428, 168259, 99861, 284054, 12445, 181808, 330457,
    351286, 109445, 321612, 260513
]

for mid in movie_ids:
    out_path = out_dir / f"movie_id={mid}.json"

    if out_path.exists():
        print(f"Exists, skipping: {mid}")
        continue

    print(f"Fetching: {mid}")
    bundle = fetch_movie_with_credits(mid, TMDB_API_KEY)

    if bundle is None:
        print(f"Movie with Id: {mid} not found")
        continue

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(bundle, f, ensure_ascii=False)

Fetching: 0
Failed: https://api.themoviedb.org/3/movie/0 | 404 Client Error: Not Found for url: https://api.themoviedb.org/3/movie/0?api_key=9094802cf093a61d48b07e86ef4720c5
Movie with Id: 0 not found
Fetching: 299534
Fetching: 19995
Fetching: 140607
Fetching: 299536
Fetching: 597
Fetching: 135397
Fetching: 420818
Fetching: 24428
Fetching: 168259
Fetching: 99861
Fetching: 284054
Fetching: 12445
Fetching: 181808
Fetching: 330457
Fetching: 351286
Fetching: 109445
Fetching: 321612
Fetching: 260513


In [16]:
len(list(out_dir.glob("*.json"))) # Let us see the number of moies fetched

18